In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import re
from sklearn.model_selection import train_test_split

# 1. Download and Prepare Dataset
!wget https://raw.githubusercontent.com/SamirMoustafa/nmt-with-attention-for-ar-to-en/master/ara_.txt

with open('ara_.txt', 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')

eng_ara_pairs = []
for line in lines[:50000]:  # Use first 50k pairs
    if '\t' in line:
        eng, ara = line.split('\t')[:2]
        eng_ara_pairs.append([eng, ara])

def preprocess_text(text):
    text = re.sub(r"([?.!,¿])", r" \1 ", text)
    text = re.sub(r'[" "]+', " ", text)
    text = re.sub(r"[^a-zA-Z?.!,¿ء-ي]+", " ", text)
    text = text.strip()
    return '<start> ' + text + ' <end>'

word_pairs = [[preprocess_text(eng), preprocess_text(ara)] for eng, ara in eng_ara_pairs]
train_pairs, test_pairs = train_test_split(word_pairs, test_size=0.2, random_state=42)

# 2. Tokenization
eng_tokenizer = keras.preprocessing.text.Tokenizer(filters='')
eng_tokenizer.fit_on_texts([eng for eng, ara in train_pairs])
eng_vocab_size = len(eng_tokenizer.word_index) + 1

ara_tokenizer = keras.preprocessing.text.Tokenizer(filters='')
ara_tokenizer.fit_on_texts([ara for eng, ara in train_pairs])
ara_vocab_size = len(ara_tokenizer.word_index) + 1

def tokenize_pairs(pairs):
    eng_texts = [eng for eng, ara in pairs]
    ara_texts = [ara for eng, ara in pairs]

    eng_sequences = eng_tokenizer.texts_to_sequences(eng_texts)
    ara_sequences = ara_tokenizer.texts_to_sequences(ara_texts)

    eng_sequences = keras.preprocessing.sequence.pad_sequences(eng_sequences, padding='post')
    ara_sequences = keras.preprocessing.sequence.pad_sequences(ara_sequences, padding='post')

    return eng_sequences, ara_sequences

train_eng, train_ara = tokenize_pairs(train_pairs)
test_eng, test_ara = tokenize_pairs(test_pairs)

max_length_eng = train_eng.shape[1]
max_length_ara = train_ara.shape[1]

# 3. Transformer Components
class PositionalEncoding(layers.Layer):
    def __init__(self, max_position, d_model):
        super(PositionalEncoding, self).__init__()
        self.max_position = max_position
        self.d_model = d_model
        
        position = tf.range(max_position, dtype=tf.float32)[:, tf.newaxis]
        div_term = tf.exp(tf.range(0, d_model, 2, dtype=tf.float32) * 
                         (-tf.math.log(10000.0) / d_model))
        
        pe = tf.zeros((max_position, d_model))
        pe = pe.numpy()
        pe[:, 0::2] = tf.sin(position * div_term).numpy()
        pe[:, 1::2] = tf.cos(position * div_term).numpy()
        pe = pe[tf.newaxis, ...]
        
        self.pe = tf.convert_to_tensor(pe, dtype=tf.float32)

    def call(self, inputs):
        if isinstance(inputs, tf.SparseTensor):
            inputs = tf.sparse.to_dense(inputs)
        seq_len = tf.shape(inputs)[1]
        return inputs + self.pe[:, :seq_len, :]
    
    def compute_output_shape(self, input_shape):
        return input_shape

def scaled_dot_product_attention(query, key, value, mask):
    matmul_qk = tf.matmul(query, key, transpose_b=True)
    depth = tf.cast(tf.shape(key)[-1], tf.float32)
    logits = matmul_qk / tf.math.sqrt(depth)

    if mask is not None:
        logits += (mask * -1e9)

    attention_weights = tf.nn.softmax(logits, axis=-1)
    output = tf.matmul(attention_weights, value)
    return output, attention_weights

class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads, name="multi_head_attention"):
        super(MultiHeadAttention, self).__init__(name=name)
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.query_dense = layers.Dense(units=d_model)
        self.key_dense = layers.Dense(units=d_model)
        self.value_dense = layers.Dense(units=d_model)
        self.dense = layers.Dense(units=d_model)

    def split_heads(self, inputs, batch_size):
        inputs = tf.reshape(inputs, shape=(batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(inputs, perm=[0, 2, 1, 3])

    def call(self, inputs):
        query, key, value, mask = inputs['query'], inputs['key'], inputs['value'], inputs['mask']
        batch_size = tf.shape(query)[0]
        
        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)
        
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)
        
        scaled_attention, attention_weights = scaled_dot_product_attention(
            query, key, value, mask)
        
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))
        output = self.dense(concat_attention)
        return output, attention_weights

def encoder_layer(units, d_model, num_heads, dropout, name="encoder_layer"):
    inputs = keras.Input(shape=(None, d_model), name="inputs")
    padding_mask = keras.Input(shape=(1, 1, None), name="padding_mask")

    attention, _ = MultiHeadAttention(
        d_model, num_heads, name="attention")({
            'query': inputs,
            'key': inputs,
            'value': inputs,
            'mask': padding_mask
        })

    attention = layers.Dropout(dropout)(attention)
    attention = layers.LayerNormalization(epsilon=1e-6)(inputs + attention)

    outputs = layers.Dense(units=units, activation='relu')(attention)
    outputs = layers.Dense(units=d_model)(outputs)
    outputs = layers.Dropout(dropout)(outputs)
    outputs = layers.LayerNormalization(epsilon=1e-6)(attention + outputs)

    return keras.Model(inputs=[inputs, padding_mask], outputs=outputs, name=name)

def decoder_layer(units, d_model, num_heads, dropout, name="decoder_layer"):
    inputs = keras.Input(shape=(None, d_model), name="inputs")
    enc_outputs = keras.Input(shape=(None, d_model), name="encoder_outputs")
    look_ahead_mask = keras.Input(shape=(1, None, None), name="look_ahead_mask")
    padding_mask = keras.Input(shape=(1, 1, None), name='padding_mask')

    attention1, attn_weights1 = MultiHeadAttention(
        d_model, num_heads, name="attention_1")({
            'query': inputs,
            'key': inputs,
            'value': inputs,
            'mask': look_ahead_mask
        })

    attention1 = layers.LayerNormalization(epsilon=1e-6)(attention1 + inputs)

    attention2, attn_weights2 = MultiHeadAttention(
        d_model, num_heads, name="attention_2")({
            'query': attention1,
            'key': enc_outputs,
            'value': enc_outputs,
            'mask': padding_mask
        })

    attention2 = layers.Dropout(dropout)(attention2)
    attention2 = layers.LayerNormalization(epsilon=1e-6)(attention2 + attention1)

    outputs = layers.Dense(units=units, activation='relu')(attention2)
    outputs = layers.Dense(units=d_model)(outputs)
    outputs = layers.Dropout(dropout)(outputs)
    outputs = layers.LayerNormalization(epsilon=1e-6)(outputs + attention2)

    return keras.Model(
        inputs=[inputs, enc_outputs, look_ahead_mask, padding_mask],
        outputs=[outputs, attn_weights1, attn_weights2],
        name=name)

def encoder(vocab_size, num_layers, units, d_model, num_heads, dropout, name="encoder"):
    inputs = keras.Input(shape=(None,), name="inputs")
    padding_mask = keras.Input(shape=(1, 1, None), name="padding_mask")

    embeddings = layers.Embedding(vocab_size, d_model)(inputs)
    embeddings *= tf.math.sqrt(tf.cast(d_model, tf.float32))
    embeddings = PositionalEncoding(max_length_ara, d_model)(embeddings)
    outputs = layers.Dropout(dropout)(embeddings)

    for i in range(num_layers):
        outputs = encoder_layer(
            units=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            name=f"encoder_layer_{i}"
        )([outputs, padding_mask])

    return keras.Model(inputs=[inputs, padding_mask], outputs=outputs, name=name)

def decoder(vocab_size, num_layers, units, d_model, num_heads, dropout, name="decoder"):
    inputs = keras.Input(shape=(None,), name="inputs")
    enc_outputs = keras.Input(shape=(None, d_model), name="encoder_outputs")
    look_ahead_mask = keras.Input(shape=(1, None, None), name="look_ahead_mask")
    padding_mask = keras.Input(shape=(1, 1, None), name='padding_mask')

    embeddings = layers.Embedding(vocab_size, d_model)(inputs)
    embeddings *= tf.math.sqrt(tf.cast(d_model, tf.float32))
    embeddings = PositionalEncoding(max_length_eng, d_model)(embeddings)
    outputs = layers.Dropout(dropout)(embeddings)

    attention_weights = {}
    for i in range(num_layers):
        outputs, block1, block2 = decoder_layer(
            units=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            name=f"decoder_layer_{i}"
        )([outputs, enc_outputs, look_ahead_mask, padding_mask])
        
        attention_weights[f'decoder_layer{i+1}_block1'] = block1
        attention_weights[f'decoder_layer{i+1}_block2'] = block2

    return keras.Model(
        inputs=[inputs, enc_outputs, look_ahead_mask, padding_mask],
        outputs=[outputs, attention_weights],
        name=name)

# 4. Transformer Model
def transformer(input_vocab_size, target_vocab_size, num_layers, units, d_model, num_heads, dropout, name="transformer"):
    # Input layers
    inputs = keras.Input(shape=(None,), name="inputs")
    dec_inputs = keras.Input(shape=(None,), name="dec_inputs")

    # Encoder padding mask
    enc_padding_mask = layers.Lambda(
        lambda x: tf.cast(tf.math.equal(x, 0), tf.float32)[:, tf.newaxis, tf.newaxis, :],
        name='enc_padding_mask')(inputs)

    # Decoder look ahead mask
    def create_look_ahead_mask(size):
        mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
        return mask[tf.newaxis, tf.newaxis, :, :]  # (1, 1, seq_len, seq_len)

    look_ahead_mask = layers.Lambda(
        lambda x: create_look_ahead_mask(tf.shape(x)[1]),
        name='look_ahead_mask')(dec_inputs)

    # Decoder padding mask
    dec_padding_mask = layers.Lambda(
        lambda x: tf.cast(tf.math.equal(x, 0), tf.float32)[:, tf.newaxis, tf.newaxis, :],
        name='dec_padding_mask')(inputs)

    # Encoder
    enc_outputs = encoder(
        vocab_size=input_vocab_size,
        num_layers=num_layers,
        units=units,
        d_model=d_model,
        num_heads=num_heads,
        dropout=dropout,
    )([inputs, enc_padding_mask])

    # Decoder
    dec_outputs, _ = decoder(
        vocab_size=target_vocab_size,
        num_layers=num_layers,
        units=units,
        d_model=d_model,
        num_heads=num_heads,
        dropout=dropout,
    )([dec_inputs, enc_outputs, look_ahead_mask, dec_padding_mask])

    # Final output
    outputs = layers.Dense(target_vocab_size, name="outputs")(dec_outputs)

    return keras.Model(
        inputs=[inputs, dec_inputs],
        outputs=outputs,
        name=name)

# 5. Training Setup
NUM_LAYERS = 4
D_MODEL = 128
NUM_HEADS = 8
UNITS = 512
DROPOUT = 0.1
EPOCHS = 20
BATCH_SIZE = 64

def loss_function(y_true, y_pred):
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')(y_true, y_pred)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

class CustomSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

learning_rate = CustomSchedule(D_MODEL)
optimizer = keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

transformer_model = transformer(
    input_vocab_size=ara_vocab_size,
    target_vocab_size=eng_vocab_size,
    num_layers=NUM_LAYERS,
    units=UNITS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dropout=DROPOUT)

transformer_model.compile(optimizer=optimizer, loss=loss_function)

def create_dataset(ara_sequences, eng_sequences, batch_size):
    decoder_input = eng_sequences[:, :-1]
    decoder_target = eng_sequences[:, 1:]

    dataset = tf.data.Dataset.from_tensor_slices((
        {'inputs': ara_sequences, 'dec_inputs': decoder_input},
        decoder_target
    ))

    dataset = dataset.cache()
    dataset = dataset.shuffle(buffer_size=len(ara_sequences))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_dataset = create_dataset(train_ara, train_eng, BATCH_SIZE)
test_dataset = create_dataset(test_ara, test_eng, BATCH_SIZE)

# 6. Training
history = transformer_model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset
)

# 7. Translation Function
def translate(sentence):
    # Preprocess input
    sentence = preprocess_text(sentence)
    inputs = [ara_tokenizer.word_index.get(word, 1) for word in sentence.split(' ')]  # 1 for <unk>
    inputs = keras.preprocessing.sequence.pad_sequences([inputs], maxlen=max_length_ara, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    # Initialize output with start token
    output = tf.expand_dims([eng_tokenizer.word_index['<start>']], 0)

    # Generate translation
    for i in range(max_length_eng):
        # Create padding mask for encoder
        enc_padding_mask = tf.cast(tf.math.equal(inputs, 0), tf.float32)
        enc_padding_mask = enc_padding_mask[:, tf.newaxis, tf.newaxis, :]
        
        # Create look-ahead and padding masks for decoder
        look_ahead_mask = 1 - tf.linalg.band_part(tf.ones((tf.shape(output)[1], tf.shape(output)[1])), -1, 0)
        look_ahead_mask = look_ahead_mask[tf.newaxis, tf.newaxis, :, :]  # Add batch and head dimensions
        dec_padding_mask = tf.cast(tf.math.equal(inputs, 0), tf.float32)
        dec_padding_mask = dec_padding_mask[:, tf.newaxis, tf.newaxis, :]
        
        # Get predictions
        predictions = transformer_model(
            {'inputs': inputs, 'dec_inputs': output}, 
            training=False)
        
        # Select the last word
        predictions = predictions[:, -1:, :]
        predicted_id = tf.cast(tf.argmax(predictions, axis=-1), tf.int32)
        
        # Return if end token is predicted
        if predicted_id == eng_tokenizer.word_index['<end>']:
            break
            
        # Concatenate the predicted word to the output
        output = tf.concat([output, predicted_id], axis=-1)

    # Convert to text
    predicted_sentence = eng_tokenizer.sequences_to_texts(output.numpy())[0]
    predicted_sentence = predicted_sentence.replace('<start>', '').replace('<end>', '').strip()
    return predicted_sentence

# Example translations
print("Translation:", translate("مرحبا كيف حالك؟"))
print("Translation:", translate("ما هو اسمك؟"))
print("Translation:", translate("انا احب التعلم"))